# Solution 1: data, baseline, splits, learning curves, label noise
One way to do tasks 0-3 and 5. Try them yourself first; compare afterwards.
Needs `uv add pandas` on top of the packages from the task description.

In [ ]:
import os, sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))   # gestures/load.py (run from gestures/sol/)
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from load import load, features, GESTURES, FEATURE_NAMES

# Download the class data once:  curl -o gestures.zip "<export link from the projector>"
SRC = os.environ.get("GESTURES_SRC") or "gestures.zip"
X, y_str, person, meta = load(SRC, return_meta=True)
y = np.array([GESTURES.index(g) for g in y_str])
people = np.unique(person)
n_people = len(people)
print(X.shape, f"{n_people} people")
pd.crosstab(person, y_str).reindex(columns=GESTURES, fill_value=0)

## Data check: sample rates, platforms, gravity
Idle takes should show |acc| ≈ 9.8. The gravity direction should agree between platforms (upright phone: y ≈ +9.8; tilted back: y and z positive).

In [ ]:
m = pd.DataFrame(meta).assign(person=person, gesture=y_str)
m["platform"] = np.where(m.source == "phyphox", "phyphox", np.where(m.ios, "iOS", "android/other"))
m["acc_mean_x"], m["acc_mean_y"], m["acc_mean_z"] = X[:, :, :3].mean(1).T
m["acc_norm"] = np.linalg.norm(X[:, :, :3], axis=2).mean(1)
m.groupby(["platform"]).agg(takes=("id", "size"), hz=("raw_hz", "median"), x=("acc_mean_x", "mean"),
                            y=("acc_mean_y", "mean"), z=("acc_mean_z", "mean"), norm=("acc_norm", "mean")).round(2)

## Task 1: look at the takes

In [ ]:
t = np.arange(X.shape[1]) / 50
fig, axs = plt.subplots(2, 6, figsize=(20, 6), sharex=True)
for j, g in enumerate(GESTURES):
    for i in np.where(y == j)[0][:5]:
        axs[0, j].plot(t, X[i, :, :3], lw=0.8)
        axs[1, j].plot(t, X[i, :, 3:], lw=0.8)
    axs[0, j].set_title(g)
axs[0, 0].set_ylabel("acc x,y,z  m/s²"); axs[1, 0].set_ylabel("gyro x,y,z  rad/s")
plt.tight_layout()

In [ ]:
# one gesture across people: same label, different styles
g = "left"
ps = people[:6]
fig, axs = plt.subplots(1, len(ps), figsize=(4 * len(ps), 3), sharey=True, squeeze=False)
for ax, p in zip(axs[0], ps):
    for i in np.where((person == p) & (y_str == g))[0][:5]:
        ax.plot(t, X[i, :, 0], lw=0.8)
    ax.set_title(f"{p}: {g}, acc x")
plt.tight_layout()

## Task 1/2: features, random split vs per person

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold, StratifiedKFold, cross_val_score, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.neighbors import KNeighborsClassifier, NearestNeighbors

F = features(X)
models = {
    "logreg": lambda: make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
    "random forest": lambda: RandomForestClassifier(300, n_jobs=-1, random_state=0),
    "kNN (k=5, scaled)": lambda: make_pipeline(StandardScaler(), KNeighborsClassifier(5)),
}
min_class = np.bincount(y, minlength=6).min()
rows = []
for name, mk in models.items():
    if min_class >= 2:
        cv = StratifiedKFold(min(5, min_class), shuffle=True, random_state=0)
        rows.append((name, "random", cross_val_score(mk(), F, y, cv=cv).mean()))
    if n_people >= 2:
        rows.append((name, "by person", cross_val_score(mk(), F, y, cv=GroupKFold(min(5, n_people)), groups=person).mean()))
    else:
        print("only one person so far: no per-person split possible")
pd.DataFrame(rows, columns=["model", "split", "accuracy"]).round(3)

## Task 2: whose recording is the nearest neighbor?
Random split: mostly the same person, so the model can match the person rather than the gesture. Per person: never the same person (by construction), and accuracy drops.

In [ ]:
from sklearn.model_selection import train_test_split

def nn_same_person(tr, te):
    sc = StandardScaler().fit(F[tr])
    idx = NearestNeighbors(n_neighbors=1).fit(sc.transform(F[tr])).kneighbors(
        sc.transform(F[te]), return_distance=False)[:, 0]
    nn = tr[idx]
    return (person[nn] == person[te]).mean(), (y[nn] == y[te]).mean(), nn

rows, example = [], None
if min_class >= 2:
    tr, te = train_test_split(np.arange(len(X)), test_size=0.25, stratify=y, random_state=0)
    same, acc, nn = nn_same_person(tr, te)
    rows.append(("random", same, acc))
    example = pd.DataFrame({"test take": [f"{person[i]} / {y_str[i]}" for i in te[:8]],
                            "its nearest neighbor": [f"{person[j]} / {y_str[j]}" for j in nn[:8]]})
if n_people >= 2:
    res = [nn_same_person(tr, te)[:2] for tr, te in GroupKFold(min(5, n_people)).split(F, y, person)]
    rows.append(("by person", *np.mean(res, 0)))
display(pd.DataFrame(rows, columns=["split", "NN is the same person", "1-NN accuracy"]).round(3))
example

In [ ]:
# confusion matrix, per-person split if possible
cv = GroupKFold(min(5, n_people)) if n_people >= 2 else StratifiedKFold(min(5, max(2, min_class)), shuffle=True, random_state=0)
pred = cross_val_predict(models["random forest"](), F, y, cv=cv, groups=person if n_people >= 2 else None)
ConfusionMatrixDisplay.from_predictions(y, pred, labels=list(range(6)), display_labels=GESTURES, normalize="true", values_format=".2f");

## Task 3: learning curves (random forest, test people fixed)
Needs ≥ 2 people.

In [ ]:
rep_idx = np.zeros(len(X), int)
for p in people:
    for j in range(6):
        idx = np.where((person == p) & (y == j))[0]
        rep_idx[idx] = np.arange(len(idx))

def rf_acc(keep):
    accs, ns = [], []
    for tr, te in GroupKFold(min(5, n_people)).split(F, y, person):
        tr = tr[keep(tr)]
        if len(np.unique(y[tr])) < 2: continue
        clf = RandomForestClassifier(200, n_jobs=-1, random_state=0).fit(F[tr], y[tr])
        accs.append((clf.predict(F[te]) == y[te]).mean()); ns.append(len(tr))
    return np.mean(accs), np.mean(ns)

if n_people >= 2:
    reps = [(r, *rf_acc(lambda tr: rep_idx[tr] < r)) for r in [1, 2, 3, 5, 10, 15] if r <= rep_idx.max() + 1]
    max_k = n_people - int(np.ceil(n_people / min(5, n_people)))
    ppl = []
    for k in sorted({1, 2, 3, 5, 8, 12, 20, max_k}):
        if k > max_k: continue
        res = []
        for seed in range(3):
            r_ = np.random.default_rng(seed)
            res.append(rf_acc(lambda tr: np.isin(person[tr], r_.choice(u := np.unique(person[tr]), min(k, len(u)), replace=False))))
        ppl.append((k, *np.mean(res, 0)))
    plt.plot([n for _, _, n in reps], [a for _, a, _ in reps], "o-", label="more reps per person")
    plt.plot([n for _, _, n in ppl], [a for _, a, _ in ppl], "s-", label="more people")
    plt.xlabel("training takes"); plt.ylabel("accuracy on unseen people"); plt.ylim(0, 1); plt.legend()

## Task 5: label noise (random forest)

In [ ]:
rng = np.random.default_rng(0)
cv = GroupKFold(min(5, n_people)) if n_people >= 2 else StratifiedKFold(min(5, max(2, min_class)), shuffle=True, random_state=0)
res = []
for frac in [0, 0.1, 0.2, 0.4]:
    accs = []
    for tr, te in cv.split(F, y, person if n_people >= 2 else None):
        ytr = y[tr].copy()
        flip = rng.random(len(tr)) < frac
        ytr[flip] = (ytr[flip] + rng.integers(1, 6, flip.sum())) % 6     # a random *other* class
        clf = RandomForestClassifier(200, n_jobs=-1, random_state=0).fit(F[tr], ytr)
        accs.append((clf.predict(F[te]) == y[te]).mean())
    res.append((frac, np.mean(accs)))
pd.DataFrame(res, columns=["flipped", "test accuracy"]).round(3)